# COLAB 2: Processing multi-subject datasets with high-performance computing

## Course architecture

Data acquisition ➡️ Training an ML model ➡️ Visualizing results ➡️ Documenting research

## Instructions for the student
In this lab, you will train a machine learning (ML) model (convolutional neural network, CNN) on temporal EEG data recorded from multiple subjects and channels.

Your objective is to pass the explicit **"AI Prompt Challenges"** directly to an AI coding assistant, evaluate the structural properties of the returned functions, and paste them into the designated code boundaries.

## Module 0: Setting up an environment on FABRIC

**Student AI prompt challenge**

Copy and paste the code block from the cell below and the following text block into your AI assistant:

> "Give me commands I can use in the ubuntu terminal to (1) install anaconda, (2) create a conda environment, (3) activate that environment, (4) install numpy, pandas, matplotlib, pytorch (GPU-compatible)."

## Module 1: Data acquisition

We will continue working with the [EEG Emotion Recognition dataset](https://www.kaggle.com/datasets/wajahat1064/emotion-recognition-using-eeg-and-computer-games?resource=download), acquiring data from all 28 subjects. The cells below download the full dataset from Kaggle.

In [1]:
import kagglehub
import os

# Download from kaggle
path = kagglehub.dataset_download("wajahat1064/emotion-recognition-using-eeg-and-computer-games")
print("Dataset downloaded to:", path)

path = os.path.join(path,
                     "Dataset - Emotion Recognition data Based on EEG Signals and Computer Games",
                     "Database for Emotion Recognition System Based on EEG Signals and Various Computer Games - GAMEEMO",
                     "GAMEEMO"
                    )

100%|██████████| 1.66G/1.66G [01:21<00:00, 22.0MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/wajahat1064/emotion-recognition-using-eeg-and-computer-games/versions/2


**Student AI prompt challenge**

The `subjects` variable below should hold subject identifiers from the dataset (S01-S28). Design a prompt and define this variable with the AI.

Create a Python list variable named `subjects` containing subject identifier strings from "S01" to "S28", zero-padded to two digits.

In [2]:
#TODO: your code here
# Generates ['S01', 'S02', ..., 'S28']
subjects = [f"S{i:02d}" for i in range(1, 29)]

In [3]:
print(len(subjects))

28


The functions below read data from a single subjects and combine the resulting dataset into an array of size (subjects x game_types x channels x time).

In [4]:
import numpy as np
import pandas as pd

def read_subject(subject, path):
    assert len(os.listdir(path)) == 4

    subject_recordings = []
    for game_idx in range(4):
        # Read channel x time data
        csv_filename = f'{subject}G{game_idx + 1}AllChannels.csv'
        df = pd.read_csv(os.path.join(path, csv_filename))
        df.drop("Unnamed: 14", axis = 1, inplace = True)

        # Shape into a channels x time array
        subject_recordings.append(df.to_numpy().T)

    # Shape into game_types x channels x time
    subject_recordings = np.stack(subject_recordings)

    return subject_recordings

def read_subjects(subjects, path):
    all_recordings = []
    for subject in subjects:
        subject_path = os.path.join(path, f"({subject})", "Preprocessed EEG Data", ".csv format")
        data = read_subject(subject, subject_path)
        all_recordings.append(data)

    # Subjects x game_types x channels x time
    all_recordings = np.stack(all_recordings)
    return all_recordings

In [5]:
data = read_subjects(subjects, path)

FileNotFoundError: [Errno 2] No such file or directory: '/root/.cache/kagglehub/datasets/wajahat1064/emotion-recognition-using-eeg-and-computer-games/versions/2/Dataset - Emotion Recognition data Based on EEG Signals and Computer Games/Database for Emotion Recognition System Based on EEG Signals and Various Computer Games - GAMEEMO/GAMEEMO/(S26)/Preprocessed EEG Data/.csv format'

**Student AI prompt challenge**

As you can see, there is an error when reading data of one of the subjects. Design a prompt to list the contents of the subject's directory and compare the contents with antoher subject (e.g., subject S01). What is the problem?

In [7]:
#TODO: your code here: list the contents of S26's directory and compare with S01
import os
from pathlib import Path

# Base directory where subject folders reside
base_path = Path(
    "/root/.cache/kagglehub/datasets/wajahat1064/"
    "emotion-recognition-using-eeg-and-computer-games/versions/2/"
    "Dataset - Emotion Recognition data Based on EEG Signals and Computer Games/"
    "Database for Emotion Recognition System Based on EEG Signals and Various Computer Games - GAMEEMO/"
    "GAMEEMO"
)

# 1. Check how the S01 and S26 directories are actually named
print("Subject folders found matching '26' or '01':")
for p in sorted(base_path.glob("*")):
    if "01" in p.name or "26" in p.name:
        print(f"  - {p.name}")

# 2. Inspect subdirectories inside S01 vs S26
for sub in ["S01", "(S01)", "S26", "(S26)"]:
    target = base_path / sub
    if target.exists():
        print(f"\nContents of {target.name}:")
        for item in target.rglob("*"):
            if item.is_file():
                print(f"    {item.relative_to(target)}")

Subject folders found matching '26' or '01':
  - (S01)
  - (S26)

Contents of (S01):
    SAM Ratings/G3.pdf
    SAM Ratings/G2.pdf
    SAM Ratings/G4.pdf
    SAM Ratings/G1.pdf
    Raw EEG Data/.mat format/S01G3AllRawChannels.mat
    Raw EEG Data/.mat format/S01G1AllRawChannels.mat
    Raw EEG Data/.mat format/S01G4AllRawChannels.mat
    Raw EEG Data/.mat format/S01G2AllRawChannels.mat
    Raw EEG Data/.csv format/S01G4AllRawChannels.csv
    Raw EEG Data/.csv format/S01G1AllRawChannels.csv
    Raw EEG Data/.csv format/S01G2AllRawChannels.csv
    Raw EEG Data/.csv format/S01G3AllRawChannels.csv
    Preprocessed EEG Data/.mat format/S01G2AllChannels.mat
    Preprocessed EEG Data/.mat format/S01G4AllChannels.mat
    Preprocessed EEG Data/.mat format/S01G3AllChannels.mat
    Preprocessed EEG Data/.mat format/S01G1AllChannels.mat
    Preprocessed EEG Data/.csv format/S01G1AllChannels.csv
    Preprocessed EEG Data/.csv format/S01G4AllChannels.csv
    Preprocessed EEG Data/.csv format/S01G2Al

**Student AI prompt challenge**

Prompt AI to modify the list of subjects to exclude subject 26 (but still include subjects 1-25 and 27-28).

In [9]:
#TODO: your code here
# Generates S01 through S25 and S27 through S28
subjects = [f"S{i:02d}" for i in range(1, 29) if i != 26]

In [10]:
data = read_subjects(subjects, path)

# Subjects x game_types x channels x time
print(data.shape)

(27, 4, 14, 38252)


## Module 2: Training an ML model

We are going to train a convolutional neural network (CNN) to predict the game type from the temporal data across channels of all available subjects.

**Student AI prompt challenge**

Copy and paste the code block from the cell below and the following text block into your AI assistant:

> "I am working with a dataset of size subjects x conditions x channels x time, my actual shape is (27, 4, 14, 38252). The data are currently stored as a numpy array. Write pytorch code that trains a simple CNN on that data to predict the condition (i.e., multiclass classification over 4 classes). Treat each subject x condition as a separate sample and make the CNN layers 1d (i.e., we convolve across time). The first layer should take 14 channels as evident from my data shape. Also leave 3 subjects out for validation (i.e., train only on 24 subjects) and add some one-time testing step (compute accuracy across the 3 subjects). Track loss and training accuracy so that I can plot them / save them later."

In [15]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Set device and seeds for reproducibility
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(42)
np.random.seed(42)

# Assume `data` is your numpy array of shape: (27, 4, 14, 38252)
# data = np.load("eeg_data.npy")

n_subjects, n_conditions, n_channels, n_time = 27, 4, 14, 38252

# Subject-level split: 24 subjects for training, 3 for holdout test/validation
train_raw = data[:24]  # (24, 4, 14, 38252)
test_raw = data[24:]  # (3, 4, 14, 38252)


def prepare_tensors(subset):
    n_sub, n_cond, ch, t = subset.shape
    # Flatten subject and condition dimensions -> (n_samples, channels, time)
    X = subset.reshape(n_sub * n_cond, ch, t)
    # Ground truth labels: condition index [0, 1, 2, 3] repeated per subject
    y = np.tile(np.arange(n_cond), n_sub)
    return torch.tensor(X, dtype=torch.float32), torch.tensor(
        y, dtype=torch.long
    )


X_train, y_train = prepare_tensors(train_raw)  # (96, 14, 38252)
X_test, y_test = prepare_tensors(test_raw)  # (12, 14, 38252)

train_loader = DataLoader(
    TensorDataset(X_train, y_train), batch_size=8, shuffle=True
)
test_loader = DataLoader(
    TensorDataset(X_test, y_test), batch_size=4, shuffle=False
)


# 1D Temporal CNN across time points
class EEG1DCNN(nn.Module):

    def __init__(self, in_channels: int = 14, num_classes: int = 4):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: Large temporal receptive field
            nn.Conv1d(in_channels, 32, kernel_size=64, stride=2, padding=32),
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.MaxPool1d(kernel_size=4, stride=4),
            nn.Dropout(p=0.25),
            # Block 2
            nn.Conv1d(32, 64, kernel_size=32, stride=2, padding=16),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.MaxPool1d(kernel_size=4, stride=4),
            nn.Dropout(p=0.25),
            # Block 3
            nn.Conv1d(64, 128, kernel_size=16, stride=1, padding=8),
            nn.BatchNorm1d(128),
            nn.GELU(),
            nn.AdaptiveAvgPool1d(1),  # Reduces time dimension to 1
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(p=0.5),
            nn.Linear(64, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.features(x))


model = EEG1DCNN(in_channels=14, num_classes=4).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)

epochs = 40
history = {"train_loss": [], "train_acc": []}

# Training Loop
for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100.0

    history["train_loss"].append(epoch_loss)
    history["train_acc"].append(epoch_acc)

    if epoch % 5 == 0 or epoch == epochs:
        print(
            f"Epoch [{epoch:02d}/{epochs:02d}] - Loss: {epoch_loss:.4f} | Acc: {epoch_acc:.2f}%"
        )

# One-time Evaluation on Left-out Subjects (N = 3)
model.eval()
test_loss = 0.0
test_correct = 0
test_total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        test_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        test_correct += (preds == labels).sum().item()
        test_total += labels.size(0)

test_acc = (test_correct / test_total) * 100.0

print("-" * 45)
print(f"Left-out Subjects (N = 3) Accuracy: {test_acc:.2f}%")
print(f"Left-out Subjects Average Loss:      {test_loss / test_total:.4f}")
print("-" * 45)

Epoch [05/40] - Loss: 1.3461 | Acc: 36.46%
Epoch [10/40] - Loss: 1.1342 | Acc: 43.75%
Epoch [15/40] - Loss: 0.9413 | Acc: 61.46%
Epoch [20/40] - Loss: 0.8892 | Acc: 62.50%
Epoch [25/40] - Loss: 0.7018 | Acc: 68.75%
Epoch [30/40] - Loss: 0.7330 | Acc: 75.00%
Epoch [35/40] - Loss: 0.4910 | Acc: 81.25%
Epoch [40/40] - Loss: 0.7126 | Acc: 73.96%
---------------------------------------------
Left-out Subjects (N = 3) Accuracy: 66.67%
Left-out Subjects Average Loss:      0.9453
---------------------------------------------


## Module 3: Visualizing results

**Student AI prompt challenge**

Design a prompt to plot the saved train and validation losses and accuracies.

In [13]:
#TODO: your code here
import matplotlib.pyplot as plt


def plot_learning_curves(history: dict):
    """Plots training and validation loss and accuracy curves across epochs.

    Expected history keys: 'train_loss', 'val_loss', 'train_acc', 'val_acc'
    """
    epochs = range(1, len(history["train_loss"]) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), dpi=120)

    # 1. Loss Subplot
    axes[0].plot(
        epochs,
        history["train_loss"],
        label="Train Loss",
        color="#1f77b4",
        lw=2,
    )
    if "val_loss" in history and history["val_loss"]:
        axes[0].plot(
            epochs,
            history["val_loss"],
            label="Val Loss",
            color="#ff7f0e",
            lw=2,
            ls="--",
        )
    axes[0].set_title("Cross-Entropy Loss", fontsize=12, fontweight="bold")
    axes[0].set_xlabel("Epoch", fontsize=11)
    axes[0].set_ylabel("Loss", fontsize=11)
    axes[0].grid(True, linestyle=":", alpha=0.6)
    axes[0].legend(frameon=True)

    # 2. Accuracy Subplot
    axes[1].plot(
        epochs, history["train_acc"], label="Train Acc", color="#1f77b4", lw=2
    )
    if "val_acc" in history and history["val_acc"]:
        axes[1].plot(
            epochs,
            history["val_acc"],
            label="Val Acc",
            color="#ff7f0e",
            lw=2,
            ls="--",
        )
    axes[1].axhline(
        y=25.0,
        color="gray",
        linestyle=":",
        linewidth=1.2,
        label="Chance Level (25%)",
    )
    axes[1].set_title("Classification Accuracy", fontsize=12, fontweight="bold")
    axes[1].set_xlabel

## Module 4: Documenting research

**Student AI prompt challenge**
Design a prompt to save the resulting  train and validation losses and accuracies in a `.csv` file.

In [14]:
#TODO: your code here
from pathlib import Path
import pandas as pd


def save_training_metrics(
    history: dict, filename: str = "training_metrics.csv"
) -> Path:
    """Converts training metric history dictionary to a pandas DataFrame and exports to CSV."""
    df = pd.DataFrame(history)
    df.insert(0, "epoch", range(1, len(df) + 1))

    output_path = Path(filename)
    df.to_csv(output_path, index=False)
    print(f"Metrics saved to {output_path.resolve()}")
    return output_path


# Usage:
# save_training_metrics(history, "cnn_eeg_training_metrics.csv")

## Module 5: Additional prompt challenges

1. **Beat the Baseline:** Use AI to modify the CNN architecture (number of layers, kernel sizes, channels, pooling) and improve validation accuracy over the provided baseline.
2. **Try Different Optimizers:** Replace Adam with SGD, RMSprop, or AdamW. Compare convergence speed and final accuracy.
3. **Learning Rate Tuning:** Use AI to implement a learning rate scheduler (e.g., StepLR, CosineAnnealingLR, ReduceLROnPlateau) and determine whether it improves performance.
4. **Normalization Challenge:** Normalize the EEG channels (e.g., z-score using only the training subjects). Compare results with the unnormalized data.
5. **Confusion Matrix:** Compute and visualize the confusion matrix for the held-out subjects. Which conditions are most frequently confused?